[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_latent = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, x_q, x_kv):
        B, S_q, d_model = x_q.shape
        _, S_kv, _ = x_kv.shape
        Q = self.W_q(x_q).view(B, S_q, self.num_heads, -1).transpose(1, 2)    # B, N, S_q, d_latent
        K = self.W_k(x_kv).view(B, S_kv, self.num_heads, -1).transpose(1, 2)  # B, N, S_kv, d_latent
        V = self.W_v(x_kv).view(B, S_kv, self.num_heads, -1).transpose(1, 2)  # B, N, S_kv, d_latent
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_latent)  # B, N, S_q, S_kv
        print(f"{scores.shape=}")
        weights = torch.softmax(scores, dim=-1)  # B, N, S_q, S_kv
        attn = torch.matmul(weights, V)   # B, N, S_q, d_latent
        print(f"{attn.shape=}")
        return self.W_o(attn.transpose(1, 2).contiguous().view(B, S_q, -1))


        pass  # Q from x_q, K/V from x_kv, no causal mask

In [4]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

scores.shape=torch.Size([2, 4, 6, 10])
attn.shape=torch.Size([2, 4, 6, 16])
Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
scores.shape=torch.Size([2, 4, 6, 10])
attn.shape=torch.Size([2, 4, 6, 16])
  ✅ [1/4] Output shape (2.0ms)
scores.shape=torch.Size([1, 2, 3, 20])
attn.shape=torch.Size([1, 2, 3, 16])
  ✅ [2/4] Q and KV different lengths (1.0ms)
scores.shape=torch.Size([1, 2, 4, 6])
attn.shape=torch.Size([1, 2, 4, 16])
scores.shape=torch.Size([1, 2, 4, 6])
attn.shape=torch.Size([1, 2, 4, 16])
  ✅ [3/4] No causal mask — all KV affects all Q (34.3ms)
scores.shape=torch.Size([1, 2, 4, 6])
attn.shape=torch.Size([1, 2, 4, 16])
  ✅ [4/4] Gradient flow (19.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (57.0ms total)
  Progress saved. Run status() to see your dashboard.

